# Graphical exploration with DSLD
Explore group distributions, multivariate patterns, and possible conditional disparities. These examples use bundled data and reproducible subsets. Plotly functions return interactive figures; static R plots return displayed images. Install the optional R package `bnlearn` to run the structure-discovery example.


In [ ]:
import tempfile
from pathlib import Path
from dsldPy import (dsldPyScatterPlot3D, dsldPyFreqPCoord, dsldPyConditDisparity,
                    dsldPyConfounders, dsldPyDensitybyS, dsldPyIamb)


In [ ]:
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from dsldPy import dsld_Rpy2_RDataframeToPandas

ro.r["set.seed"](2026)
bundled = ro.Environment()
def load_bundled(name):
    importr("utils").data(name, package="dsld", envir=bundled)
    return dsld_Rpy2_RDataframeToPandas(bundled[name])

census = load_bundled("svcensus").sample(n=800, random_state=2026)
records = load_bundled("compas1")
records = records.loc[records["race"].isin(["Caucasian", "African-American"])]
records = records[["age", "decile_score", "priors_count", "sex", "race", "two_year_recid"]]
records["race"] = records["race"].cat.remove_unused_categories()
records = records.sample(n=800, random_state=2026)


## 1. Three-dimensional scatterplot


In [ ]:
dsldPyScatterPlot3D(census, ["wageinc", "wkswrkd", "age"], "gender")


## 2. Frequency-based parallel coordinates


In [ ]:
# Keep the axes focused and readable; compare these measures by sex.
parallel_data = records[["age", "decile_score", "priors_count", "sex"]]
dsldPyFreqPCoord(parallel_data, m=50, sName="sex", k=15, klm=50)


## 3. Conditional disparity
For a binary outcome, the default qeKNN backend uses the first factor level as its positive class in this plotting interface.


In [ ]:
dsldPyConditDisparity(records, "two_year_recid", "race", "age",
                       condits=["priors_count <= 4", "decile_score >= 4"], minS=30)


## 4. Conditional-independence structure discovery
The learned structure is exploratory and requires assumptions for causal interpretation. The plot is written to a temporary directory for this example.


In [ ]:
with tempfile.TemporaryDirectory() as directory:
    dsldPyIamb(census[["age", "wkswrkd", "gender"]], file=str(Path(directory) / "iamb.png"))


## 5. Group frequencies and densities


In [ ]:
dsldPyConfounders(census[["age", "educ", "gender"]], sName="gender")


In [ ]:
density_figure = dsldPyDensitybyS(census, cName="wageinc", sName="gender")
# The returned figure can be customized or saved with write_html().
